# RAG Pipeline
This notebook covers the **complete RAG pipeline** in one place:

| Step | What happens |
|---|---|
| **Step 0** | Setup — libraries, config, load FAISS + metadata |
| **Step 1** | Question Embedding |
| **Step 2** | Vector Retrieval from FAISS |
| **Step 3** | Relevance Threshold Check |
| **Step 4** | Prompt Construction |
| **Step 5** | LLM Call |
| **Step 6** | Full Pipeline Function (`rag_answer`) |
| **Step 7** | Interactive Mode — ask your own question |
| **Step 8** | Batch Testing — 8 test questions |
| **Step 9** | Evaluation — Hit Rate + MRR |
| **Step 10** | Results Table + Save to JSON |

---

> **Before running this notebook, make sure:**
> 1. All 4 data pipeline notebooks (`01` → `04`) have been fully run
> 2. `data/embeddings/faiss_index.bin` exists
> 3. `data/embeddings/chunks_with_metadata.json` exists
> 4. Your `.env` file has `GROQ_API_KEY=your_key_here` in the project root

In [1]:
import os
import json
import numpy as np
import pandas as pd
import faiss
from groq import Groq  
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

print("Imports successful")

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful


In [2]:
# Load API key from .env file
load_dotenv("../.env")

# Configuration parameters
EMBEDDINGS_DIR = Path("../../data/embeddings")
RAG_DIR        = Path("../../data/rag")

EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "llama-3.1-8b-instant"   # CHANGED: llama3-8b-8192 decommissioned, official replacement
MAX_TOKENS     = 300

TOP_K          = 3      # Number of chunks to retrieve per question
SIM_THRESHOLD  = 0.3    # Min similarity score — below this = no answer

# CHANGED: Replaced OLLAMA_OPTIONS with GROQ client init.
# Groq API is a cloud inference service — no local model loading, no KV-cache warmup.
# The client reads GROQ_API_KEY automatically from the environment.
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

RAG_DIR.mkdir(parents=True, exist_ok=True)

# Verify setup and print configuration
groq_key = os.getenv("GROQ_API_KEY")
print(f"Groq API key       : {'Loaded' if groq_key else 'NOT FOUND — check your .env file'}")  # CHANGED
print(f"Embedding model    : {EMBED_MODEL}")
print(f"LLM model          : {LLM_MODEL}")
print(f"Top-K retrieval    : {TOP_K}")
print(f"Similarity threshold: {SIM_THRESHOLD}")

Groq API key       : Loaded
Embedding model    : sentence-transformers/all-MiniLM-L6-v2
LLM model          : llama-3.1-8b-instant
Top-K retrieval    : 3
Similarity threshold: 0.3


## Load Embedding Model + FAISS Index + Chunk Metadata

We load three things here:
- **Embedding model** — same model used in `embedding.ipynb` (question and chunks must be in the same vector space)
- **FAISS index** — the vector store built from all chunk embeddings
- **Chunk metadata** — text + source info for every chunk stored in the index

In [3]:
# Load Embedding Model
print("Loading embedding model...")

# CHANGED: Added device='cpu' to avoid slow CUDA init probing when no GPU is present.
# Added show_progress_bar=False to suppress tqdm output that adds overhead per encode() call.
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")
embed_model.max_seq_length = 128   # CHANGED: Cap token length — FAQ chunks are short; default 256 wastes compute.
print(f"Embedding model loaded | Dimensions: {embed_model.get_sentence_embedding_dimension()}")

# Load FAISS Index 
faiss_path = "../../data/embeddings/faiss_index.bin"
index = faiss.read_index(str(faiss_path))
print(f"\nFAISS index loaded")
print(f"   Vectors in index : {index.ntotal}")
print(f"   Dimensions       : {index.d}")

# Load Chunk Metadata 
meta_path = Path("../../data/embeddings/chunks_with_metadata.json")
with open(meta_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"\nChunk metadata loaded")
print(f"   Total chunks     : {len(chunks)}")

# Sanity check — index and metadata must be in sync
assert index.ntotal == len(chunks), (
    f"Mismatch: FAISS has {index.ntotal} vectors but metadata has {len(chunks)} chunks. "
    "Re-run 04_embedding.ipynb."
)
print("\nIndex and metadata are in sync — ready to retrieve")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1226.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded | Dimensions: 384

FAISS index loaded
   Vectors in index : 207
   Dimensions       : 384

Chunk metadata loaded
   Total chunks     : 207

Index and metadata are in sync — ready to retrieve


---
## Question Embedding + Vector Retrieval

**How it works:**
1. The user question is converted to a dense vector using the same embedding model as the chunks
2. FAISS performs an inner product search — since all vectors are L2-normalized, this equals **cosine similarity**
3. The top-K most similar chunk indices + scores are returned
4. Scores are already in range `[0, 1]` — higher = more relevant

In [4]:
# Retrieval Function - Given a question, return top-K most similar chunks with metadata
"""
Embed the question and retrieve top-K most similar chunks from FAISS.

Args:
    question : User's question string.
    top_k    : Number of chunks to return.

Returns:
    List of chunk dicts, each with keys:
    rank, chunk_id, bank_name, source_file, page_number, text, similarity
"""

def retrieve_chunks(question: str, top_k: int = TOP_K) -> list[dict]:
    # Step 1 — Embed the question (L2-normalize to match index vectors)
    # CHANGED: Added batch_size=1 and show_progress_bar=False to skip tqdm setup
    # overhead on every single-question call. convert_to_numpy=True avoids a
    # redundant tensor→numpy copy that the default path does internally.
    question_vec = embed_model.encode(
        [question],
        normalize_embeddings=True,
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32)

    # Step 2 — FAISS search
    # IndexFlatIP + normalized vectors → scores = cosine similarities
    scores, indices = index.search(question_vec, top_k)

    # Step 3 — Build result list with metadata
    retrieved = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk = chunks[idx].copy()
        chunk["rank"]       = rank
        chunk["similarity"] = round(float(score), 4)
        retrieved.append(chunk)

    return retrieved


# Quick Test
print("Testing retrieval...\n")
test_chunks = retrieve_chunks("How to open a bank account in Pakistan?")

for r in test_chunks:
    print(f"Rank {r['rank']} | Similarity: {r['similarity']} | {r['bank_name']}")
    print(f"  Source : {r['source_file']} | Page {r['page_number']}")
    print(f"  Text   : {r['text'][:200]}...")
    print()

Testing retrieval...

Rank 1 | Similarity: 0.6935 | Meezan Bank
  Source : Meezan-Bank-FAQs-Roshan-Digital-Account.pdf | Page 2
  Text   : an Digital Account.
18. How do I apply for Internet Banking facility for my Meezan Roshan Digital account?
Please refer to detailed guidelines for Internet banking facilities available at https://www....

Rank 2 | Similarity: 0.6862 | Habib Bank Limited (HBL)
  Source : HBL-Work-Conventional-Accounts.pdf | Page 3
  Text   : Requirements to open an account: To open the account you will 
need to satisfy some identiﬁcation requirements as per regulatory 
instructions and banks' internal policies. These may include providing...

Rank 3 | Similarity: 0.6575 | Meezan Bank
  Source : Meezan-Bank-FAQs-Roshan-Digital-Account.pdf | Page 1
  Text   : FAQs of Meezan Roshan Digital Account 
1.
What is Meezan Roshan Digital Account?
Meezan Roshan Digital Account is a unique opportunity for Individual Non-Resident Pakistanis (NRPs) to open a bank
acco...



---
## Relevance Threshold Check

**Why this step?**  
If a user asks something completely unrelated to our dataset (e.g., *"What is the capital of France?"*), we should NOT call the LLM with irrelevant chunks — it wastes tokens and risks a hallucinated answer.

**Logic:**
- Best similarity score **≥ threshold (0.3)** → proceed to prompt + LLM
- Best similarity score **< threshold (0.3)** → return fallback, skip LLM entirely

In [5]:
# Relevance Check Function - Determine if retrieved chunks are relevant based on similarity
"""
Check if retrieved chunks pass the similarity threshold.

Returns:
    dict with keys:
    - is_relevant     (bool)
    - best_similarity (float)
    - status          (str): 'relevant' | 'below_threshold'
"""

def check_relevance(retrieved: list[dict]) -> dict:
    if not retrieved:
        return {"is_relevant": False, "best_similarity": 0.0, "status": "no_results"}

    best_similarity = max(c["similarity"] for c in retrieved)
    is_relevant     = best_similarity >= SIM_THRESHOLD

    return {
        "is_relevant":     is_relevant,
        "best_similarity": best_similarity,
        "status":          "relevant" if is_relevant else "irrelevant"
    }


# Test both cases
in_scope  = retrieve_chunks("What is the cash withdrawal limit at Meezan Bank?")
out_scope = retrieve_chunks("What is the capital of France?")

check_in  = check_relevance(in_scope)
check_out = check_relevance(out_scope)

print(f"In-scope query  | Best sim: {check_in['best_similarity']:<6} | Status: {check_in['status']}")
print(f"Out-scope query | Best sim: {check_out['best_similarity']:<6} | Status: {check_out['status']}")

In-scope query  | Best sim: 0.6465 | Status: relevant
Out-scope query | Best sim: 0.1368 | Status: irrelevant


---
## Prompt Generation

We build a **grounded RAG prompt** that:
- Assigns LLM a clear role as a Pakistani banking assistant
- Injects the retrieved chunks as **labelled context** (bank + source + page)
- Strictly restricts LLM to answer **only from the context**
- Provides an explicit fallback phrase for unanswerable questions
- Preserves all numbers, limits, and amounts from the source

In [6]:
# Prompt Generation Function - Create a grounded RAG prompt using retrieved chunks as context
"""
Build a grounded RAG prompt using retrieved chunks as context.
The LLM is instructed to answer ONLY from this context.

Args:
    question        : User's question string.
    retrieved_chunks: List of retrieved chunk dicts from retrieve_chunks().
    
Returns:
    Complete prompt string ready to send to the LLM.
"""

def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    # Format each chunk with a labelled source header
    context_parts = []
    for chunk in retrieved_chunks:
        header = (
            f"[Source {chunk['rank']}: {chunk['bank_name']} "
            f"| {chunk['source_file']} | Page {chunk['page_number']}]"
        )
        context_parts.append(f"{header}\n{chunk['text']}")

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant for Pakistani banking customers.
Answer the question ONLY based on the context documents provided below.
If the answer is not in the context, say exactly: "I don't have enough information to answer this from the available documents."
Do NOT make up any information. Be concise and accurate.
If amounts, limits, or numbers are mentioned in the context, always include them in your answer.

=== CONTEXT DOCUMENTS ===
{context}
=== END OF CONTEXT ===

QUESTION: {question}

ANSWER"""

    return prompt


# ── Preview a sample prompt ──────────────────────────────────────
sample_question = "What is the late payment fee for Alfalah personal loan?"
sample_chunks   = retrieve_chunks(sample_question)
sample_prompt   = build_prompt(sample_question, sample_chunks)

print("📋 SAMPLE PROMPT PREVIEW (first 1000 chars):")
print("=" * 60)
print(sample_prompt[:1000])
print("...")

📋 SAMPLE PROMPT PREVIEW (first 1000 chars):
You are a helpful assistant for Pakistani banking customers.
Answer the question ONLY based on the context documents provided below.
If the answer is not in the context, say exactly: "I don't have enough information to answer this from the available documents."
Do NOT make up any information. Be concise and accurate.
If amounts, limits, or numbers are mentioned in the context, always include them in your answer.

=== CONTEXT DOCUMENTS ===
[Source 1: Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf | Page 2]
FAQs – Personal Loan 

1st year 
2nd year 
3rd year onwards 
Not allowed 
8% of the paid amount 
5% of the paid amount 
 
10. Can I settle my loan before the end of term? 
 
Yes, you may give a 30 day prior written notice to the Bank if you wish for early settlement 
of your facility. Based on prior written consent of the Bank on your request, you may fully 
pre-pay your loan. 
 
11. Is there an early settlement fee? 
 
Yes, 
1st year 
2

---
## LLM Call — Llama 3 8B (Groq)

We send the constructed prompt to **Llama 3 8B** hosted on **Groq** and get a grounded answer back.

> Make sure your Groq API key is set before executing this cell:  
> 1. Get a free key → https://console.groq.com  
> 2. Add `GROQ_API_KEY=your_key_here` to your `.env` file  
> 3. No local model download or Ollama server needed — Groq runs in the cloud

In [7]:
def call_llm(prompt: str) -> str:
    """
    Send the RAG prompt to Llama 3 8B via Groq API and return the answer.
    Uses the Groq Python SDK — requires GROQ_API_KEY in .env.

    Args:
        prompt: The complete RAG prompt string.

    Returns:
        Llama's answer as a plain string.
    """
    # CHANGED: Replaced ollama.chat() with groq_client.chat.completions.create().
    # Groq runs inference on custom LPU hardware — response times are typically
    # under 2 seconds vs 30-40 minutes with local Ollama cold-start.
    # temperature=0.1 keeps answers factual and grounded to the context.
    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=MAX_TOKENS,
        temperature=0.1,       # CHANGED: Low temperature = factual, grounded answers
        top_p=0.9,             # CHANGED: Slightly restricted nucleus sampling
    )

    return response.choices[0].message.content.strip()  # CHANGED: Groq response format


# Connectivity test 
print("Testing LLM connection...")
test_response = call_llm("Say exactly the words: LLM connection successful")
print(f"LLM Response: {test_response}")

Testing LLM connection...
LLM Response: LLM connection successful.


---
## Full RAG Pipeline (`rag_answer`)

All steps combined into one function:

```
question
    → embed question
    → FAISS search → top-K chunks
    → similarity threshold check
    → build prompt
    → call Groq
    → return structured result
```

In [8]:
# Fallback answer for when no relevant information is found in the retrieved chunks
FALLBACK_ANSWER = "No relevant information found in the available banking documents."


# Complete RAG Pipeline Function - From question to final answer with sources
"""
Complete RAG Pipeline:
  1. Embed the question
  2. Retrieve top-K chunks from FAISS
  3. Check similarity threshold
  4. Build the grounded prompt
  5. Call Groq LLM
  6. Return structured result

Args:
    question (str) : User's question.
    verbose  (bool): Print step-by-step output.

Returns:
    dict with keys: question, answer, sources, best_similarity, status
"""

def rag_answer(question: str, verbose: bool = True) -> dict:
    if verbose:
        print(f"\n{'='*65}")
        print(f"QUESTION: {question}")
        print(f"{'='*65}")

    # Retrieve chunks
    retrieved = retrieve_chunks(question, top_k=TOP_K)

    if verbose:
        print(f"\nRetrieved {len(retrieved)} chunks:")
        for c in retrieved:
            print(f"   [{c['similarity']}] {c['bank_name']} | {c['source_file']}")

    # Relevance threshold check
    relevance       = check_relevance(retrieved)
    best_similarity = relevance["best_similarity"]

    if not relevance["is_relevant"]:
        if verbose:
            print(f"\nBest similarity ({best_similarity}) is below threshold ({SIM_THRESHOLD})")
            print("Not enough relevant context found — skipping LLM call.")
        return {
            "question":        question,
            "answer":          FALLBACK_ANSWER,
            "sources":         [],
            "best_similarity": best_similarity,
            "status":          "below_threshold"
        }

    # Build prompt
    prompt = build_prompt(question, retrieved)

    # Call LLM
    if verbose:
        print(f"\nSending to Groq ({LLM_MODEL})...")  # CHANGED: label updated to Groq

    answer  = call_llm(prompt)
    sources = list(set(c["source_file"] for c in retrieved))

    if verbose:
        print(f"\nANSWER:")
        print(f"   {answer}")
        print(f"\nSOURCES USED:")
        for s in sources:
            print(f"   • {s}")

    return {
        "question":        question,
        "answer":          answer,
        "sources":         sources,
        "best_similarity": best_similarity,
        "status":          "answered",
        # CHANGED: Store retrieved chunks in result so the evaluation step can
        # reuse them directly — avoids a redundant embed+search call per question.
        "_retrieved":      retrieved,
    }


print("rag_answer() function defined and ready.")

rag_answer() function defined and ready.


---
## Interactive Mode

Change the question below and run the cell to test any question against the pipeline.

In [9]:
# Change this question and run the cell
MY_QUESTION = "What is the minimum age requirement for Alfalah personal loan?"

result = rag_answer(MY_QUESTION, verbose=True)


QUESTION: What is the minimum age requirement for Alfalah personal loan?

Retrieved 3 chunks:
   [0.6798] Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf
   [0.5834] Bank Alfalah | Bank-Alfalah-FAQs-Personal-Loan.pdf
   [0.5528] Meezan Bank | Meezan-Bank-FAQs-Salaried.pdf

Sending to Groq (llama-3.1-8b-instant)...

ANSWER:
   The minimum age requirement for Alfalah personal loan is 21 years for salaried individuals and 65 years for SEB/SEP individuals.

SOURCES USED:
   • Meezan-Bank-FAQs-Salaried.pdf
   • Bank-Alfalah-FAQs-Personal-Loan.pdf


---
## Batch Testing

Running **8 test questions** covering all banks and topics in the dataset, plus one out-of-scope question to verify the threshold filter works correctly.

In [10]:
# 8 TEST QUESTIONS — all banks + one out-of-scope 
# Format: { "question": str, "expected_source": str | None }
# expected_source is used in Step 9 for Hit Rate / MRR evaluation

TEST_QUESTIONS = [
    # HBL — Home Remittance
    {"question": "What is the cash transaction limit for HBL home remittance service?",
     "expected_source": "HBL-FAQs-Home-Remittance"},

    # Meezan — Roshan Digital Account
    {"question": "What documents are needed to open a Meezan Roshan Digital Account?",
     "expected_source": "Meezan-Bank-FAQs-Roshan-Digital-Account"},

    # Meezan — Roshan Apna Ghar (solar financing)
    {"question": "Is net metering included in Meezan Bank solar panel financing?",
     "expected_source": "Meezan-Bank-FAQs-Roshan-Apna-Ghar"},

    # Bank Alfalah — Personal Loan
    {"question": "What is the late payment charge on Alfalah personal loan?",
     "expected_source": "Bank-Alfalah-FAQs-Personal-Loan"},

    # ABL — Digital Banking
    {"question": "Can a foreign national register for ABL myABL digital banking?",
     "expected_source": "ABL"},

    # State Bank of Pakistan
    {"question": "What is the withholding tax rate on Pakistan Investment Bonds?",
     "expected_source": "State-Bank"},

    # HBL Islamic
    {"question": "Can Islamic banks charge penalty for late payment?",
     "expected_source": "HBL-Islamic"},

    # Out-of-scope — must trigger below_threshold
    {"question": "What is the capital of France?",
     "expected_source": None},
]

print(f"📋 {len(TEST_QUESTIONS)} test questions loaded")
print(f"   {len([q for q in TEST_QUESTIONS if q['expected_source']])} in-scope questions")
print(f"   {len([q for q in TEST_QUESTIONS if not q['expected_source']])} out-of-scope question")

📋 8 test questions loaded
   7 in-scope questions
   1 out-of-scope question


In [11]:
# Run all test questions through the RAG pipeline and collect results
all_results = []

for question_item in TEST_QUESTIONS:
    result = rag_answer(question_item["question"], verbose=True)
    result["expected_source"] = question_item["expected_source"]
    all_results.append(result)
    print()


QUESTION: What is the cash transaction limit for HBL home remittance service?

Retrieved 3 chunks:
   [0.7264] Habib Bank Limited (HBL) | HBL-FAQs-Home-Remittance.pdf
   [0.6951] Habib Bank Limited (HBL) | HBL-FAQs-Home-Remittance.pdf
   [0.5842] Habib Bank Limited (HBL) | HBL-Work-Conventional-Accounts.pdf

Sending to Groq (llama-3.1-8b-instant)...

ANSWER:
   The cash transaction limit for HBL home remittance service is PKR 500,000/- per transaction via HBL's Cash Over the Counter (CoC) service.

SOURCES USED:
   • HBL-FAQs-Home-Remittance.pdf
   • HBL-Work-Conventional-Accounts.pdf


QUESTION: What documents are needed to open a Meezan Roshan Digital Account?

Retrieved 3 chunks:
   [0.7664] Meezan Bank | Meezan-Bank-FAQs-Digital-Account.pdf
   [0.7134] Meezan Bank | Meezan-Bank-FAQs-Roshan-Digital-Account.pdf
   [0.6817] Meezan Bank | Meezan-Bank-FAQs-Roshan-Digital-Account.pdf

Sending to Groq (llama-3.1-8b-instant)...

ANSWER:
   According to the context documents, no additional

---
## Evaluation — Hit Rate + MRR

We evaluate **retrieval quality** on the 7 in-scope questions using two metrics:

| Metric | What it measures |
|---|---|
| **Hit Rate** | Did the correct source appear anywhere in the top-K retrieved chunks? |
| **MRR** (Mean Reciprocal Rank) | How high was the correct source ranked? Rewards rank 1 more than rank 3. |

MRR formula: `MRR = mean(1 / rank_of_first_hit)` — a perfect score is `1.0` (always rank 1).

In [12]:
# Only evaluate in-scope questions (those with an expected source)
in_scope_results = [r for r in all_results if r["expected_source"] is not None]

hits             = 0
reciprocal_ranks = []

for r in in_scope_results:
    # CHANGED: Reuse the _retrieved chunks already stored in the result dict
    # instead of calling retrieve_chunks() again — eliminates a redundant
    # embed+search call for every in-scope question during evaluation.
    retrieved_for_eval = r.get("_retrieved") or retrieve_chunks(r["question"], top_k=TOP_K)
    retrieved_sources  = [c["source_file"] for c in retrieved_for_eval]
    expected           = r["expected_source"]

    # Find the rank of the first hit
    hit_rank = None
    for rank, src in enumerate(retrieved_sources, 1):
        if expected.lower() in src.lower():
            hit_rank = rank
            break

    hit = hit_rank is not None
    rr  = 1 / hit_rank if hit else 0.0

    hits += int(hit)
    reciprocal_ranks.append(rr)

    # Attach metrics back to result for the summary table
    r["hit"]      = hit
    r["hit_rank"] = hit_rank
    r["rr"]       = rr

    status = f"Hit @ rank {hit_rank}" if hit else "Miss"
    print(f"{status:22s} | {r['question'][:60]}")

hit_rate = hits / len(in_scope_results)
mrr      = float(np.mean(reciprocal_ranks))

print(f"\n{'='*50}")
print(f"Hit Rate (top-{TOP_K}) : {hit_rate:.2%}  ({hits}/{len(in_scope_results)})")
print(f"MRR               : {mrr:.4f}")
print(f"{'='*50}")

Hit @ rank 1           | What is the cash transaction limit for HBL home remittance s
Hit @ rank 2           | What documents are needed to open a Meezan Roshan Digital Ac
Miss                   | Is net metering included in Meezan Bank solar panel financin
Hit @ rank 1           | What is the late payment charge on Alfalah personal loan?
Hit @ rank 1           | Can a foreign national register for ABL myABL digital bankin
Hit @ rank 1           | What is the withholding tax rate on Pakistan Investment Bond
Miss                   | Can Islamic banks charge penalty for late payment?

Hit Rate (top-3) : 71.43%  (5/7)
MRR               : 0.6429


---
## Results Summary Table + Save to JSON

In [13]:
# Build summary dataframe for all questions (including out-of-scope) with key metrics
rows = []
for r in all_results:
    is_out_of_scope = r["expected_source"] is None

    if is_out_of_scope:
        retrieval_col = "N/A (out-of-scope)"
    else:
        retrieval_col = f"Hit @ rank {r.get('hit_rank')}" if r.get("hit") else "Miss"

    rows.append({
        "Question"       : r["question"][:55] + "...",
        "RAG Status"     : "Answered" if r["status"] == "answered" else "No Info",
        "Best Sim"       : r["best_similarity"],
        "Retrieval"      : retrieval_col,
        "Sources Used"   : ", ".join(s.replace(".pdf", "") for s in r["sources"]) or "—",
        "Answer Preview" : r["answer"][:75] + "..."
    })

df_results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 230)

print("RAG PIPELINE — COMPLETE TEST RESULTS")
print("=" * 110)
print(df_results.to_string(index=False))

RAG PIPELINE — COMPLETE TEST RESULTS
                                                  Question RAG Status  Best Sim          Retrieval                                                              Sources Used                                                                 Answer Preview
What is the cash transaction limit for HBL home remitta...   Answered    0.7264       Hit @ rank 1                  HBL-FAQs-Home-Remittance, HBL-Work-Conventional-Accounts The cash transaction limit for HBL home remittance service is PKR 500,000/-...
What documents are needed to open a Meezan Roshan Digit...   Answered    0.7664       Hit @ rank 2 Meezan-Bank-FAQs-Digital-Account, Meezan-Bank-FAQs-Roshan-Digital-Account According to the context documents, no additional documents are required to...
Is net metering included in Meezan Bank solar panel fin...   Answered    0.7864               Miss                                                 Meezan-Bank-FAQs-Salaried                  No, net metering 

In [16]:
# Save results to JSON for further analysis
save_records = []
for r in all_results:
    save_records.append({
        "question":        r["question"],
        "answer":          r["answer"],
        "sources":         r["sources"],
        "best_similarity": r["best_similarity"],
        "status":          r["status"],
        "hit":             r.get("hit"),
        "hit_rank":        r.get("hit_rank"),
        "rr":              r.get("rr"),
        "expected_source": r.get("expected_source")
    })

# Save results to JSON (in test directory)
output_path = RAG_DIR / "rag_test_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(save_records, f, ensure_ascii=False, indent=2)

print(f"Results saved to : {output_path}")
print(f"   Total questions  : {len(save_records)}")
print(f"       Answered       : {sum(1 for r in save_records if r['status'] == 'answered')}")
print(f"       Below threshold: {sum(1 for r in save_records if r['status'] == 'irrelevant')}")

Results saved to : ..\..\data\rag\rag_test_results.json
   Total questions  : 8
       Answered       : 7
       Below threshold: 0


---
## Pipeline Summary

| Component | Choice | Reason |
|---|---|---|
| Embedding Model | `all-MiniLM-L6-v2` | Fast, free, strong semantic quality for English FAQ |
| Vector Store | FAISS `IndexFlatIP` | No server needed — cosine sim via dot product on normalized vecs |
| LLM | Llama 3 8B (Groq) | Free tier available, runs on Groq LPU — responses in ~1-2 seconds |
| Similarity Metric | Cosine Similarity | Standard for semantic search |
| Threshold | 0.3 | Filters completely irrelevant out-of-domain queries |
| Chunk Retrieval | Top-3 | Best balance of context richness vs noise |
| Prompt Strategy | Context-only grounding | Prevents LLM hallucination |

---

### 📁 Files Generated After Running This Notebook
```
data/
├── embeddings/
│   ├── faiss_index.bin              ← from embedding.ipynb
│   └── chunks_with_metadata.json    ← from embedding.ipynb
└── rag/
    └── rag_test_results.json        ← NEW: saved from this notebook
```